# Boom Phase 2.1 Combined
แผนทดลอง Current baseline → Boom integration

Notebook นี้ใช้ phase22.py ตรวจ baseline และ tune.py รัน inference จริง อ่านผลการรันที่ส่วนท้าย ไม่มีผลจำลอง

In [1]:
from pathlib import Path
import sys, json
start = Path.cwd().resolve()
choices = [start, *start.parents]
HERE = next((q for p in choices for q in (p, p / "Boom/phase2.2") if (q / "phase22.py").is_file()), None)
if HERE is None:
    raise FileNotFoundError("เปิด notebook จาก repo หรือ Boom/phase2.2")
sys.path.insert(0, str(HERE))
import phase22 as ex
print(json.dumps(ex.baseline(), ensure_ascii=False, indent=2))

{
  "root": "/Users/thawaphorn/Pee4/Term1/DSI490/DSI490-Sign2Say",
  "version": "v7.2",
  "revision": "5b31439e4f21c66b4818ca2acdfc79de322c7fec",
  "working_tree_dirty": true,
  "spotting": {
    "lens": [
      8,
      12,
      16,
      20,
      24,
      28,
      32,
      36,
      40,
      44,
      48
    ],
    "stride": 4,
    "min_inside": 0.6,
    "delta": 0.0,
    "act_thr": 0.5,
    "mu_null": 5.0,
    "csls_alpha": 0.0,
    "one_pass": true,
    "exact_rescore": false,
    "lam": 6.0,
    "gap_cost": 0.5,
    "min_len": 14,
    "beta": 5.0,
    "freq_gamma": 0.0,
    "tempo": {
      "ref_len": 16,
      "min_factor": 1.2,
      "max_factor": 2.5,
      "min_segments": 2
    },
    "post": {
      "min_frames": 10,
      "null_gap_max": 0.1,
      "p_sign_min": 0.65,
      "repeat_penalty": true
    },
    "grammar_rerank_weight": 0.5
  },
  "effective_tau_accept": 0.8,
  "inference_run": false
}


**อ่านผล:** version/revision มาจากไฟล์จริง ถ้า working_tree_dirty เป็น true ต้องบันทึก diff และ artifact hashes ก่อนรันเทียบคะแนน ไม่ถือว่า revision อย่างเดียวระบุ baseline ได้ครบ

## เป้าหมาย
นำแนวคิดจากสามสายมาทดลองบนโค้ดที่มีจริง รักษาคำที่ทำเร็ว ลดคำแทรก แยกท่าคล้าย และยืนยันคำได้เหมาะสม ไม่อ้างว่าเป็น v8.1 และไม่คัดลอกคะแนนจากรายงานมาเป็นผลทดลอง


## สถาปัตยกรรม
วิดีโอ → MediaPipe / Standard Schema → Tagger → window encoder + bank scoring → Decoder → post filters / grammar → calibrated status → ข้อความ

ส่วนทดลองเพิ่มเติมคือ hand descriptor จากรูปนิ้วและการเคลื่อนไหวของ landmark เทียบ descriptor ของคลิปคำจริงใน bank เสริม ก่อนรวมกับคะแนน encoder ต้องตรวจคุณภาพมือและปรับสเกลคะแนนบน tune ช่องที่ตรวจไม่ได้เป็น unknown ไม่ตีความว่าเป็นมือกำ และ cosine ไม่ใช่ probability


## ลำดับทดลอง
1. ล็อก revision, config, encoder, tagger, bank และ vocabulary เป็น baseline เดียวกัน บันทึก hash ก่อนรัน
2. รัน baseline บน tune ที่มีจริงที่ 1×/1.5×/2× แยกประโยคจริงกับประโยคสังเคราะห์
3. ทดลอง A/B/B2 เกณฑ์ p_sign_min, D/E/E2 เกณฑ์ตัดช่วง, F exact_rescore ทีละตัวแปร
4. C ตรวจ effective tau_accept ก่อน การเปลี่ยน accept_min อย่างเดียวอาจไม่มีผลเพราะ constructor ใช้ max กับ calibration; เก็บ threshold ที่ใช้จริงทุกครั้ง
5. G พัฒนาตัวกรองมือหายแยก flag ไม่ใช้ confidence ที่ interpolate แล้วแทน raw observed rate โดยไม่ระบุข้อจำกัด
6. Hand-shape และ grammar เป็นรอบแยกหลัง baseline พร้อม ไม่ hardcode คำ และไม่บังคับน้ำหนัก 70:30; ตรวจ grammar leak ก่อนเลือกค่า
7. H รวมเฉพาะค่าที่ผ่านแล้วทดสอบ tune ซ้ำ การผ่านแยกตัวไม่รับประกันว่ารวมแล้วผ่าน
8. ล็อก config ก่อนเปิด test รอบสุดท้าย และประเมิน Eval-Test เพื่อรายงานเท่านั้น


## เกณฑ์ผ่าน
ไม่มี tune set ใด WER แย่ลงเกิน 0.003; WER เฉลี่ยดีขึ้นมากกว่า 0.002; accepted precision ≥0.90 ทุกชุด หากไม่มีคำ accepted ให้เป็น N/A และไม่ถือว่าผ่าน


## รายงานที่ต้องได้เมื่อรันจริง
- WER และ S/D/I ทั้ง macro ต่อคลิปและ corpus WER แยกนิยาม
- Ordered Recall@1/5 จาก alignment เดียวกัน; ระบุ denominator และไม่ใช่นับคำที่โผล่ตรงไหนก็ได้
- Segment F1 แบบ 1:1 ที่ IoU≥0.3 เฉพาะชุดที่มีขอบเขตอ้างอิง ระบุถ้าเป็น automatic annotation; ชื่อวิดีโออย่างเดียววัด F1 ไม่ได้
- Count error = abs(predicted count-reference count), accepted correct/count/precision และ acceptance coverage
- ECE ของ p_correct เทียบ correctness พร้อม bin definition; ห้ามใช้ cosine เป็น probability
- Latency แยก cold load, extraction และ warm end-to-end บนเครื่องเดียวกัน ไม่ใช้เวลา replay cached แทน
- Paired bootstrap 10,000 ครั้งของ delta WER; cluster ตาม signer หากมี ID และจำนวน cluster เพียงพอ; CI คร่อมศูนย์รายงานว่า ไม่มีนัยสำคัญ
- ภาพแต่ละ segment, เวลาเริ่ม–จบ, GT alignment, Top-5, cosine/support, p_correct, hand score (ถ้ามี), final score และ Accept/Uncertain/Reject
- ตารางช่วยแก้/ทำแย่ลง, confusion pairs และตัวอย่างคำสั้นหรือคำใกล้ปากที่หาย โดยแยกหลักฐานจากข้อสันนิษฐาน


## วินัยข้อมูล
ไม่ retrain ไม่แก้ bank/models ต้นฉบับ ทุกฟีเจอร์ใหม่ default ปิด ใช้ config/descriptor bank แยก ผล Eval-Test ห้ามใช้เลือกค่า Boom 6 คลิปเคยใช้พัฒนาแล้วจึงไม่ใช่ unseen test; signer overlap ยังต้องตรวจด้วย metadata และ IDs ที่ยืนยันได้


## ไฟล์และสถานะ
เปิด boom_phase2_2.ipynb เพื่ออ่านแผนและตรวจ config ปัจจุบันผ่าน phase22.py ไฟล์ experiment_plan.json เป็นแผนที่ยังไม่เปิดใช้งาน ไม่ใช่ config ผู้ชนะ
**มี runner จริงสำหรับ baseline/A/B/B2/C/D/E/E2/F แล้ว ดูผลและสถานะท้าย Notebook; G/handshape/grammar/H ยังไม่รัน** ไม่เปลี่ยนเว็บหรือ production จึงไม่ต้อง rollback


In [2]:
experiment = ex.plan()
print(json.dumps(experiment, ensure_ascii=False, indent=2))

{
  "name": "Boom Phase 2.2",
  "baseline": "current checkout; record actual revision, never assume v8.1",
  "enabled": false,
  "production_enable": false,
  "trials": {
    "baseline": {},
    "A": {
      "post.p_sign_min": 0.55
    },
    "B": {
      "post.p_sign_min": 0.6
    },
    "B2": {
      "post.p_sign_min": 0.7
    },
    "C": {
      "accept_min": 0.7
    },
    "D": {
      "act_thr": 0.6
    },
    "E": {
      "min_len": 16
    },
    "E2": {
      "post.min_frames": 12
    },
    "F": {
      "exact_rescore": true
    }
  },
  "conditional_trials": {
    "C_tau": {
      "values": [
        0.85,
        0.88
      ],
      "condition": "check effective tau_accept=max(calibration tau, root accept_min) before trial"
    },
    "G": {
      "post.min_hand_frac": 0.3,
      "status": "requires implementation and raw observed-hand definition"
    },
    "handshape": {
      "enabled": false,
      "status": "requires separate descriptor bank and tune validation; no fixed

**อ่านผล:** enabled=false หมายถึงแผนยังไม่เปิดใช้ ไม่ใช่การทดลองแล้วได้คะแนนเท่ากัน การใช้งาน runner และผลจริงจะเพิ่มใน Notebook นี้เมื่อพัฒนาแต่ละขั้นเสร็จ

## รันการทดลองจริงบน tune
ใช้ tune.py และ environment ของโปรเจกต์ ผลบันทึกแยกตามเวลารัน ไม่มีการเปิด test ในสคริปต์นี้ ขณะนี้มีงานรันจาก CLI แล้ว จึงตั้ง RUN_TUNE=False เพื่อป้องกันการรันซ้ำ

In [ ]:
import subprocess
PY = ex.ROOT / "Boom/venv-sign2say/bin/python"
RUN_TUNE = False
if RUN_TUNE:
    subprocess.run([str(PY), "-B", str(HERE / "tune.py")], cwd=ex.ROOT, check=True)


## ผลจริงและเกณฑ์ผ่าน
ต้องมี DONE.json ก่อนสรุปครบทุกความเร็ว ตารางระหว่างรันยังไม่ใช้เลือกผู้ชนะ ไม่มีการเปิดใช้ config อัตโนมัติ

In [ ]:
import pandas as pd
from IPython.display import display, Image
OUT = Path(json.loads((HERE / "latest.json").read_text())["path"])
print("Complete:", (OUT / "DONE.json").exists())
for name in ["summary", "gates"]:
    file = OUT / f"{name}.csv"
    if file.exists(): display(pd.read_csv(file).round(3))
    else: print("Pending:", name)


**อ่านผล:** WER/S/D/I และ count error ใช้ทุกคำที่ไม่ใช่ transition; accepted precision นับเฉพาะคำที่ยืนยัน Recall@5 ใช้ alignment จาก top-1 ไม่ใช่ตรวจว่าคำโผล่ที่ใดก็ได้ เกณฑ์ผ่านต้องครบ 126 คลิปทั้งสามความเร็วและ precision ≥0.90 ทุกชุด

In [ ]:
if (OUT / "metrics.csv").exists():
    subprocess.run([str(PY), "-B", str(HERE / "report.py")], cwd=ex.ROOT, check=True)
    display(Image(filename=str(OUT / "tune_wer.png")))
    display(pd.read_csv(OUT / "calibration.csv").round(3))
    changes = pd.read_csv(OUT / "changes.csv")
    display(changes.sort_values("delta").head(15).round(3))


**ข้อจำกัด:** grammar คงเดิมและมีประวัติพบ reference leakage จึงเป็นผล exploratory; ยังไม่ promote production เวลาเป็น pipeline จาก pose ที่สกัดไว้ ไม่รวม extraction ค่า F1 ยังไม่มีในรอบนี้ แถว GT ที่จับคู่จาก edit alignment ไม่ใช่ human temporal annotation

## Combined experiment
เปรียบเทียบ baseline, A (`p_sign_min=.55`) และ Combined (`p_sign_min=.55`, `act_thr=.60`, `accept_min=.70`) บน tune เท่านั้น รันใหม่ทั้งสามเคสด้วย bank เดียวกัน ผลรอบเก่าเก็บไว้ตามเดิม การผ่าน tune ไม่เท่ากับผ่าน test และยังมีข้อจำกัดจาก grammar เดิม

In [ ]:
RUN_COMBINED = False  # งานเริ่มจาก CLI แล้ว ห้ามรันซ้ำระหว่างที่งานยังทำงาน
if RUN_COMBINED:
    subprocess.run([str(PY), "-B", str(HERE / "combined_run.py")], cwd=ex.ROOT, check=True)


In [ ]:
OUT = Path(json.loads((HERE / "latest.json").read_text())["path"])
for filename in ["summary.csv", "gates.csv", "paired_ci.csv"]:
    if (OUT / filename).exists():
        print(filename)
        display(pd.read_csv(OUT / filename).round(3))
print("Final report ready:", (OUT / "ANALYSIS_DONE.json").exists())


**อ่านผล:** paired CI เปรียบเทียบ Combined กับทั้ง baseline และ A แยกแต่ละความเร็ว Bootstrap 10,000 ครั้งจับคู่รายประโยค เป็น exploratory tune CI หลังเลือกค่า ไม่ใช่การยืนยันบน holdout; ไม่มีการถือว่าความเร็วต่างกันเป็นผู้ทำท่าคนละคน

## Frozen test — ห้ามใช้เลือกค่าหรือรันซ้ำ
ล็อก baseline/A/Combined จากผล tune ใน FROZEN_TEST.json ตรวจ hash ก่อนรันและมี TEST_STARTED.json ป้องกันการรันซ้ำ ชุดนี้เคยประเมินในงานก่อนหน้าและ grammar เดิมมี reference leakage จึงไม่ใช่ holdout ที่ไม่เคยเห็น ผลทุกความเร็วเป็นการประเมิน recordings เดียวกัน ไม่ใช่ผู้ทำท่าคนละคน

In [ ]:
TEST_OUT = Path(json.loads((HERE / "latest_test.json").read_text())["path"])
print("Test inference complete:", (TEST_OUT / "DONE.json").exists())
print("Analysis complete:", (TEST_OUT / "ANALYSIS_DONE.json").exists())
for filename in ["summary.csv", "paired_ci.csv", "calibration.csv"]:
    if (TEST_OUT / filename).exists():
        print(filename)
        display(pd.read_csv(TEST_OUT / filename).round(3))


**อ่านผล:** CI รวมศูนย์ = ไม่มีนัยสำคัญ ไม่รวมความเร็วต่างกันเพื่อเพิ่มจำนวนตัวอย่าง ไม่เปิด production อัตโนมัติ และไม่ย้อนกลับไปเปลี่ยนค่าหลังอ่าน test